In [1]:
import os
import json
from dotenv import load_dotenv
from google.genai import Client


load_dotenv()  # Load environment variables from .env file

True

In [8]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1

In [ ]:
# build a tool directory to map tool names to their corresponding functions


def calculator(operation: str, num1: float, num2: float) -> float:
    """
    A simple calculator function that performs basic arithmetic operations.

    Args:
        operation (str): The operation to perform ('add', 'subtract', 'multiply', 'divide').
        num1 (float): The first number.
        num2 (float): The second number.

    Returns:
        float: The result of the arithmetic operation.

    Raises:
        ValueError: If an unsupported operation is provided or division by zero occurs.
    """
    if operation == "add":
        return num1 + num2
    elif operation == "subtract":
        return num1 - num2
    elif operation == "multiply":
        return num1 * num2
    elif operation == "divide":
        if num2 == 0:
            raise ValueError("Cannot divide by zero.")
        return num1 / num2
    else:
        raise ValueError(f"Unsupported operation: {operation}")


def weather_info(city: str) -> str:
    """
    A function that provides weather information for a given city.

    Args:
        city (str): The name of the city.
        city name must be below list :
        - New York
        - Los Angeles
        - Chicago
        - Houston
        - Phoenix

    Returns:
        str: A string containing the weather information for the specified city.
    """

    # For demonstration purposes, we'll return a mock weather report.
    # In a real-world scenario, we would integrate with a weather API to fetch actual data.
    mock_weather_data = {
        "New York": "Sunny, 25°C",
        "Los Angeles": "Cloudy, 22°C",
        "Chicago": "Rainy, 18°C",
        "Houston": "Hot, 30°C",
        "Phoenix": "Sunny, 35°C",
    }

    return mock_weather_data.get(
        city, f"Weather information for {city} is not available."
    )


def get_stock_price(ticker: str) -> float:
    """
    A function that provides the current stock price for a given ticker symbol.

    Args:
        ticker (str): The stock ticker symbol (e.g., 'AAPL' for Apple, 'GOOGL' for Alphabet).

    Returns:
        float: The current stock price.
    """

    # For demonstration purposes, we'll return a mock stock price.
    # In a real-world scenario, we would integrate with a financial API to fetch actual stock prices.
    mock_stock_prices = {
        "AAPL": 150.25,
        "GOOGL": 2800.50,
        "AMZN": 3400.75,
        "MSFT": 299.99,
        "TSLA": 720.10,
    }

    return mock_stock_prices.get(ticker, f"Stock price for {ticker} is not available.")

In [ ]:
TOOL_DIRECTORY = {
    "calculator": calculator,
    "weather_info": weather_info,
    "get_stock_price": get_stock_price,
}


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "name": "calculator",
        "description": "A simple calculator that can perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "description": "The operation to perform ('add', 'subtract', 'multiply', 'divide').",
                },
                "num1": {"type": "number", "description": "The first number."},
                "num2": {"type": "number", "description": "The second number."},
            },
            "required": ["operation", "num1", "num2"],
        },
    },
    {
        "type": "function",
        "name": "weather_info",
        "description": "Provides weather information for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city. Supported cities: New York, Los Angeles, Chicago, Houston, Phoenix.",
                }
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "get_stock_price",
        "description": "Provides the current stock price for a given ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "The stock ticker symbol (e.g., 'AAPL' for Apple, 'GOOGL' for Alphabet). Supported tickers: AAPL, GOOGL, AMZN, MSFT, TSLA.",
                }
            },
            "required": ["ticker"],
        },
    },
]

In [ ]:
def tool_execute(
    name: str,
    arguments: dict,
) -> dict:
    tool = TOOL_DIRECTORY.get(name)

    if tool is None:
        return {"error": f"Unknown tool: {name}"}

    try:
        return tool(**arguments)

    except Exception as exc:
        return {"error": str(exc)}


def run_agent(user_input: str):
    print("\nUSER:")
    print(user_input)

    interaction = google_client.interactions.create(
        model=MODEL,
        input=user_input,
        tools=TOOL_DEFINITIONS,
    )

    while True:
        function_calls = [
            step for step in interaction.steps if step.type == "function_call"
        ]

        if not function_calls:
            return interaction.output_text

        function_results = []

        for function_call in function_calls:
            print("\nMODEL REQUESTED:")
            print("Tool:", function_call.name)
            print("Arguments:", function_call.arguments)

            result = tool_execute(
                name=function_call.name,
                arguments=function_call.arguments,
            )

            print("Result:", result)

            function_results.append(
                {
                    "type": "function_result",
                    "name": function_call.name,
                    "call_id": function_call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(result),
                        }
                    ],
                }
            )

        interaction = google_client.interactions.create(
            model=MODEL,
            previous_interaction_id=interaction.id,
            input=function_results,
            tools=TOOL_DEFINITIONS,
        )

In [29]:
user_query = "I want to purchase 10 shares of AAPL so please provide me the current stock price and calculate the total cost for me."

In [36]:
answer = run_agent(user_query)
print("\nFINAL ANSWER:")
print(answer)


USER:
I want to purchase 10 shares of AAPL so please provide me the current stock price and calculate the total cost for me.

MODEL REQUESTED:
Tool: get_stock_price
Arguments: {'ticker': 'AAPL'}
Result: 150.25

MODEL REQUESTED:
Tool: calculator
Arguments: {'num2': 10, 'operation': 'multiply', 'num1': 150.25}
Result: 1502.5

FINAL ANSWER:
The current stock price for AAPL is $150.25 per share. 

To purchase 10 shares, the total cost will be $1,502.50.


### Task 2

In [25]:
vague_tool_definition = {
    "type": "function",
    "name": "get_order_status",
    "description": "Gets information about an order.",
    "input_schema": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "The order ID."}},
        "required": ["order_id"],
    },
}

good_tool_definition = {
    "type": "function",
    "name": "get_order_status",
    "description": (
        "Look up the current status and estimated delivery date for a customer's order. "
        "Use this when the customer asks about an existing order's status, tracking, "
        "or delivery timing. "
        "Do NOT use this to look up or availability of product or product information or place new orders."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "order_id": {
                "type": "string",
                "description": "The order ID, formatted as ORD-XXXXXXXX",
            }
        },
        "required": ["order_id"],
    },
}

queries = [
    "Where is my order ORD-12345678?",
    "I want to place a new order for a laptop.",
    "When will my order ORD-87654321 arrive?",
    "Can you track ORD-11223344?",
    "What's the return policy for electronics?",
    "Which products do you sell?",
    "Show me the available blue jackets.",
    "How much does the iPhone 17 cost?",
    "Has my order ORD-55667788 been shipped yet?",
    "What's the current status of order ORD-99887766?",
]

expected_responses = [True, False, True, True, False, False, False, False, True, True]

In [34]:
results = {"vague tool definition": [], "good tool definition": []}

print("\n--- Testing with vague tool definition ---")
for query in queries:
    print("\nUSER QUERY:")
    print(query)

    interaction = google_client.interactions.create(
        model=MODEL,
        input=query,
        tools=[vague_tool_definition],
    )
    result = (
        True
        if any(step.type == "function_call" for step in interaction.steps)
        else False
    )
    print("Tool call request :- ", result)
    results["vague tool definition"].append(result)

print("\n--- Testing with good tool definition ---")
for query in queries:
    print("\nUSER QUERY:")
    print(query)

    interaction = google_client.interactions.create(
        model=MODEL,
        input=query,
        tools=[good_tool_definition],
    )
    result = (
        True
        if any(step.type == "function_call" for step in interaction.steps)
        else False
    )
    print("Tool call request :- ", result)
    results["good tool definition"].append(result)


--- Testing with vague tool definition ---

USER QUERY:
Where is my order ORD-12345678?
Tool call request :-  True

USER QUERY:
I want to place a new order for a laptop.
Tool call request :-  False

USER QUERY:
When will my order ORD-87654321 arrive?
Tool call request :-  True

USER QUERY:
Can you track ORD-11223344?
Tool call request :-  True

USER QUERY:
What's the return policy for electronics?
Tool call request :-  False

USER QUERY:
Which products do you sell?
Tool call request :-  False

USER QUERY:
Show me the available blue jackets.
Tool call request :-  True

USER QUERY:
How much does the iPhone 17 cost?
Tool call request :-  False

USER QUERY:
Has my order ORD-55667788 been shipped yet?
Tool call request :-  True

USER QUERY:
What's the current status of order ORD-99887766?
Tool call request :-  True

--- Testing with good tool definition ---

USER QUERY:
Where is my order ORD-12345678?
Tool call request :-  True

USER QUERY:
I want to place a new order for a laptop.
Tool ca

In [35]:
print("Expected responses :- ", expected_responses)
print("Results with vague tool definition :- ", results["vague tool definition"])
print("Results with good tool definition :- ", results["good tool definition"])

Expected responses :-  [True, False, True, True, False, False, False, False, True, True]
Results with vague tool definition :-  [True, False, True, True, False, False, True, False, True, True]
Results with good tool definition :-  [True, False, True, True, False, False, False, False, True, True]


In [36]:
for i in range(len(queries)):
    if results["vague tool definition"][i] != expected_responses[i]:
        print(
            f"Mismatch for query '{queries[i]}' with vague tool definition. Expected: {expected_responses[i]}, Got: {results['vague tool definition'][i]}"
        )

    if results["good tool definition"][i] != expected_responses[i]:
        print(
            f"Mismatch for query '{queries[i]}' with good tool definition. Expected: {expected_responses[i]}, Got: {results['good tool definition'][i]}"
        )

Mismatch for query 'Show me the available blue jackets.' with vague tool definition. Expected: False, Got: True


### Task 3

cover in task 1

### Task 4

In [37]:
def get_order_status(order_id: str):
    print(f"Executing get_order_status({order_id})")

    # Deliberately simulate a failure
    raise RuntimeError("Order service is temporarily unavailable")


def execute_tool(tool_name: str, arguments: dict):
    try:
        if tool_name == "get_order_status":
            return get_order_status(**arguments)

        raise ValueError(f"Unknown tool: {tool_name}")

    except Exception as e:
        return {"success": False, "error": str(e), "error_type": type(e).__name__}

In [ ]:
messages = [{"role": "user", "content": "What's the status of order ORD-12345678?"}]

while True:
    interaction = google_client.interactions.create(
        model=MODEL,
        previous_interaction_id=messages[-1].get("interaction_id"),
        input=messages[-1]["content"],
        tools=[good_tool_definition],
    )

    # No tool call → final answer
    if not interaction.steps or all(
        step.type != "function_call" for step in interaction.steps
    ):
        print("FINAL ANSWER:")
        print(interaction.output_text)
        break

    # Add assistant's tool request
    messages.append(interaction)

    for tool_call in interaction.steps:
        if tool_call.type == "function_call":
            print("MODEL REQUESTED:")
            print("Tool:", tool_call.name)
            print("Arguments:", tool_call.arguments)
            result = execute_tool(tool_call.name, tool_call.arguments)

            print("TOOL RESULT:")
            print(result)

            # Send tool result back to LLM
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

MODEL REQUESTED:
Tool: get_order_status
Arguments: {'order_id': 'ORD-12345678'}
Executing get_order_status(ORD-12345678)
TOOL RESULT:
{'success': False, 'error': 'Order service is temporarily unavailable', 'error_type': 'RuntimeError'}
FINAL ANSWER:
It looks like you might have pasted an error message from our system! 

I apologize for the trouble—it seems our order system is experiencing a temporary glitch right now. 

Is there anything else I can help you with in the meantime, such as looking up product information or answering other questions? If you're trying to check on an order status, please feel free to try again in a few minutes, and I'd be happy to look it up for you.


### Task 5

In [48]:
get_weather = {
    "type": "function",
    "name": "get_weather",
    "description": (
        "Get the weather information for a specified city and specific date and time."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "Name of the city."},
            "date": {
                "type": "string",
                "description": "Date and time for which the weather information is requested, in the format 'YYYY-MM-DD HH:MM'.",
            },
        },
        "required": ["city", "date"],
    },
}

get_current_weather = {
    "type": "function",
    "name": "get_current_weather",
    "description": ("Get the current weather information for a specified city."),
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string", "description": "Name of the city."}},
        "required": ["city"],
    },
}


test_queries = [
    "What's the weather in Ahmedabad at 24 sep 2026 3 PM?",
    "Tell me the weather in Mumbai right now.",
    "What's the current weather in Delhi?",
    "How is the weather in Rajkot yesterday?",
    "What's the weather like in Surat?",
    "Give me the current weather in Pune.",
    "Is the weather good in Jaipur at 10AM of 10 sep 2026?",
    "What's the weather right now in Vadodara?",
    "Tell me today's weather in Bengaluru.",
    "What's the current weather in Hyderabad?",
]

In [49]:
def select_tool(query: str):
    interaction = google_client.interactions.create(
        model=MODEL,
        input=query,
        tools=[get_weather, get_current_weather],
    )

    selected_tool = None
    tool_arguments = None

    for step in interaction.steps:
        if step.type == "function_call":
            selected_tool = step.name
            tool_arguments = step.arguments
            break

    return selected_tool, tool_arguments


results = []

for query in test_queries:
    selected, arguments = select_tool(query)

    results.append(
        {"query": query, "selected_tool": selected, "tool_arguments": arguments}
    )

    print(f"\nQUERY: {query}")
    print(f"SELECTED: {selected}")
    print(f"ARGUMENTS: {arguments}")


QUERY: What's the weather in Ahmedabad at 24 sep 2026 3 PM?
SELECTED: get_weather
ARGUMENTS: {'city': 'Ahmedabad', 'date': '2026-09-24 15:00'}

QUERY: Tell me the weather in Mumbai right now.
SELECTED: get_current_weather
ARGUMENTS: {'city': 'Mumbai'}

QUERY: What's the current weather in Delhi?
SELECTED: get_current_weather
ARGUMENTS: {'city': 'Delhi'}

QUERY: How is the weather in Rajkot yesterday?
SELECTED: get_weather
ARGUMENTS: {'city': 'Rajkot', 'date': '2024-05-18 12:00'}

QUERY: What's the weather like in Surat?
SELECTED: get_current_weather
ARGUMENTS: {'city': 'Surat'}

QUERY: Give me the current weather in Pune.
SELECTED: get_current_weather
ARGUMENTS: {'city': 'Pune'}

QUERY: Is the weather good in Jaipur at 10AM of 10 sep 2026?
SELECTED: get_weather
ARGUMENTS: {'date': '2026-09-10 10:00', 'city': 'Jaipur'}

QUERY: What's the weather right now in Vadodara?
SELECTED: get_current_weather
ARGUMENTS: {'city': 'Vadodara'}

QUERY: Tell me today's weather in Bengaluru.
SELECTED: g

### Task 6

In [55]:
def keep_looping(message: str):
    print(f"[TOOL] keep_looping called: {message}")

    return {"continue": True, "message": "Please call keep_looping again."}


loop_tool = {
    "type": "function",
    "name": "keep_looping",
    "description": (
        "Use this tool whenever the user asks you to continue looping. "
        "After receiving the result, call this tool again."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "A message to send to the looping tool.",
            }
        },
        "required": ["message"],
    },
}


def run_agent(google_client, model, user_input, tools, max_iterations=3):
    interaction_input = user_input

    for iteration in range(max_iterations):
        print(f"\n--- ITERATION {iteration + 1} ---")

        interaction = google_client.interactions.create(
            model=model,
            input=interaction_input[-1]["result"]
            if isinstance(interaction_input, list)
            else interaction_input,
            previous_interaction_id=interaction_input.get("interaction_id")
            if isinstance(interaction_input, dict)
            else None,
            tools=tools,
        )

        # Look for function calls
        function_calls = [
            step for step in interaction.steps if step.type == "function_call"
        ]

        # No function call = model produced final answer
        if not function_calls:
            return {
                "status": "completed",
                "iterations": iteration + 1,
                "response": interaction,
            }

        # Execute function calls
        tool_outputs = []

        for call in function_calls:
            if call.name == "keep_looping":
                result = keep_looping(**call.arguments)

                tool_outputs.append(
                    {
                        "type": "function_result",
                        "call_id": call.id,
                        "result": result.get("message")
                        if result.get("continue")
                        else "Stopping the loop.",
                    }
                )

        # Feed tool results into next iteration
        interaction_input = tool_outputs

    # max_iterations reached
    return {
        "status": "max_iterations_reached",
        "iterations": max_iterations,
        "response": None,
    }

In [56]:
query = """
Keep looping using the keep_looping tool.
After every tool result, call the same tool again.
Do not stop until I tell you to stop.
"""

result = run_agent(
    google_client=google_client,
    model=MODEL,
    user_input=query,
    tools=[loop_tool],
    max_iterations=5,
)

print(result)


--- ITERATION 1 ---
[TOOL] keep_looping called: Starting the loop as requested.

--- ITERATION 2 ---
[TOOL] keep_looping called: Looping as requested.

--- ITERATION 3 ---
[TOOL] keep_looping called: Continuing the loop as requested.

--- ITERATION 4 ---
[TOOL] keep_looping called: Continuing the loop as requested.

--- ITERATION 5 ---
[TOOL] keep_looping called: Calling keep_looping as requested.
{'status': 'max_iterations_reached', 'iterations': 5, 'response': None}


### Task 7

for the task log using print in above tasks